## Preprocess, trim all datasets

In [1]:
import numpy as np
import pandas as pd
import sys
import os

os.chdir(r"c:\Projects\signia-fsl-recognition")
sys.path.insert(0, r"c:\Projects\signia-fsl-recognition")
from src.landmarks.dynamic_extractor import DynamicLandmarkExtractor
from src.preprocessing.dynamic_preprocessing import preprocess_dynamic_sequence


df = pd.read_csv("data/video_manifest.csv")

print(df.head())
print(df.columns)

  video_id                                         video_path  label_id  \
0  000_001  c:\Projects\signia-fsl-recognition\dynamic_raw...         0   
1  000_002  c:\Projects\signia-fsl-recognition\dynamic_raw...         0   
2  000_003  c:\Projects\signia-fsl-recognition\dynamic_raw...         0   
3  000_004  c:\Projects\signia-fsl-recognition\dynamic_raw...         0   
4  000_005  c:\Projects\signia-fsl-recognition\dynamic_raw...         0   

          label display_label  category modality signer_id  take  enabled  \
0  GOOD MORNING  Good Morning  GREETING  dynamic   unknown     1     True   
1  GOOD MORNING  Good Morning  GREETING  dynamic   unknown     2     True   
2  GOOD MORNING  Good Morning  GREETING  dynamic   unknown     3     True   
3  GOOD MORNING  Good Morning  GREETING  dynamic   unknown     4     True   
4  GOOD MORNING  Good Morning  GREETING  dynamic   unknown     5     True   

    fps  frames  duration  
0  60.0     245  4.083333  
1  60.0     244  4.066667  
2 

In [2]:
extractor = DynamicLandmarkExtractor()

In [3]:
from tqdm import tqdm
import numpy as np

X = []
y = []
meta = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    video_path = row["video_path"]
    label_id = row["label_id"]

    try:
        # 1. extract
        sequence, metadata = extractor.extract_video(video_path)

        # 2. preprocess
        processed = preprocess_dynamic_sequence(
            sequence,
            length=30,
            scale_mode="bbox"
        )

        X.append(processed)
        y.append(label_id)
        meta.append(metadata)

    except Exception as e:
        print(f"Failed: {video_path} | {e}")

  0%|          | 0/2130 [00:00<?, ?it/s]c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
100%|██████████| 2130/2130 [11:44:26<00:00, 19.84s/it]      


In [4]:
X = np.array(X)   # (N, 30, 126)
y = np.array(y)   # (N,)

In [5]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("Classes:", len(np.unique(y)))

X shape: (2130, 30, 126)
y shape: (2130,)
Classes: 105


In [7]:
import torch

torch.save(
    {
        "X": torch.tensor(X, dtype=torch.float32),
        "y": torch.tensor(y, dtype=torch.long),
    },
    "fsl_dataset.pt"
)

In [8]:
print("Null samples:", np.sum(np.isnan(X)))
print("Zero-only samples:", np.sum(np.all(X == 0, axis=(1,2))))

Null samples: 0
Zero-only samples: 1


bad samples

In [9]:
bad_indices = np.where(
    np.all(X == 0, axis=(1,2))
)[0]

print(bad_indices)

[573]


In [10]:
idx = bad_indices[0]

print(df.iloc[idx])

sequence, metadata = extractor.extract_video(
    df.iloc[idx]["video_path"]
)

print(metadata)

video_id                                                   027_019
video_path       c:\Projects\signia-fsl-recognition\dynamic_raw...
label_id                                                        27
label                                                        EIGHT
display_label                                                Eight
category                                                    NUMBER
modality                                                   dynamic
signer_id                                                  unknown
take                                                            19
enabled                                                       True
fps                                                           60.0
frames                                                         244
duration                                                  4.066667
Name: 573, dtype: object
{'total_frames': 244, 'detected_frames': 0, 'detection_rate': 0.0}


NPZ

In [5]:
import numpy as np

data = np.load("C:\Projects\signia-fsl-recognition\dynamic_dataset.npz")

X = data["X"]
y = data["y"]

print("Shape:", X.shape)

bad = np.where(np.all(X == 0, axis=(1, 2)))[0]
print("Zero-only samples:", len(bad))
print(bad)

Shape: (2129, 30, 126)
Zero-only samples: 0
[]


PT

In [7]:
import torch

data = torch.load("C:\\Projects\\signia-fsl-recognition\\fsl_dataset.pt")

X = data["X"]

bad = torch.where(
    torch.all(X == 0, dim=(1, 2))
)[0]

print(bad)

tensor([573])


Remove the bad sample from fsl_dataset.pt

In [9]:
import torch

# Load
data = torch.load("C:\\Projects\\signia-fsl-recognition\\fsl_dataset.pt")

X = data["X"]
y = data["y"]

print(X.shape, y.shape)

# Remove bad sample
bad_idx = 573

mask = torch.ones(len(X), dtype=torch.bool)
mask[bad_idx] = False

X_clean = X[mask]
y_clean = y[mask]

print(X_clean.shape, y_clean.shape)

torch.Size([2130, 30, 126]) torch.Size([2130])
torch.Size([2129, 30, 126]) torch.Size([2129])


In [10]:
bad = torch.where(torch.all(X_clean == 0, dim=(1, 2)))[0]
print(bad)

tensor([], dtype=torch.int64)


In [11]:

torch.save(
    {
        "X": X_clean,
        "y": y_clean,
    },
    "fsl_dataset_clean.pt",
)